In [1]:
import pandas as df
from sklearn.preprocessing import StandardScaler

df = df.read_csv('KaggleV2-May-2016.csv')

#check for missing data
print("Missing values per column:")
print(df.isnull().sum())

# Drop rows with any missing values
df_clean = df.dropna()

print("\nShape before:", df.shape)
print("Shape after dropping missing values:", df_clean.shape)

# Optional: save cleaned dataset
#df_clean.to_csv("KaggleV2-May-2016.csv")

#3 Feature Extraction
#Extract the following features:

# List of features to extract
features = [
    'Gender',
    'Age',
    'Scholarship',
    'Hipertension',
    'Diabetes',
    'Alcoholism',
    'Handcap',
    'SMS_received'
]

# Extract only the selected columns
df_selected = df[features].copy()

# Display the first few rows
print(df_selected.head())


Missing values per column:
PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No-show           0
dtype: int64

Shape before: (110527, 14)
Shape after dropping missing values: (110527, 14)
  Gender  Age  Scholarship  Hipertension  Diabetes  Alcoholism  Handcap  \
0      F   62            0             1         0           0        0   
1      M   56            0             0         0           0        0   
2      F   62            0             0         0           0        0   
3      F    8            0             0         0           0        0   
4      F   56            0             1         1           0        0   

   SMS_received  
0             0  
1             0  
2             0  
3             0  
4             0  


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd

df = pd.read_csv('KaggleV2-May-2016.csv')

# Check for missing data
print("Missing values per column:")
print(df.isnull().sum())

# Drop rows with any missing values
df_clean = df.dropna()

print("\nShape before:", df.shape)
print("Shape after dropping missing values:", df_clean.shape)

# Optional: save cleaned dataset
# df_clean.to_csv("KaggleV2-May-2016.csv")

# 3 Feature Extraction
# Extract the following features:

# List of features to extract
features = [
    'Gender',
    'Age',
    'Scholarship',
    'Hipertension',
    'Diabetes',
    'Alcoholism',
    'Handcap',
    'SMS_received'
]

# Extract only the selected columns
df_selected = df[features].copy()

# Display the first few rows
print(df_selected.head())

numeric_features = ["Age", "Scholarship", "Hipertension", "Diabetes", "Alcoholism", "Handcap", "SMS_received"]
categorical_features = ["Gender"]

# Define preprocessor
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first"), categorical_features),
])

# Define X and y (target)
X = df_selected
y = df['No-show']  # or whatever your target column is in this dataset

# Split data
X_train_full, X_temp, y_train_full, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print("Train shape:", X_train_full.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape, "\n")

# SVM part (unchanged)
kernels = ["linear", "rbf", "poly", "sigmoid"]
best_kernel = None
best_val_accuracy_svm = 0.0
svm_val_results = {}
for k in kernels:
    svm_clf = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", SVC(kernel=k, C=1.0, random_state=42))
    ])
    svm_clf.fit(X_train_full, y_train_full)
    y_val_pred_svm = svm_clf.predict(X_val)
    val_acc_svm = accuracy_score(y_val, y_val_pred_svm)
    svm_val_results[k] = val_acc_svm
    print(f"SVM - Kernel = {k}, Validation Accuracy = {val_acc_svm:.4f}")
    if val_acc_svm > best_val_accuracy_svm:
        best_val_accuracy_svm = val_acc_svm
        best_kernel = k

print("\nBest SVM kernel based on validation:", best_kernel)
print("Best SVM validation accuracy:", best_val_accuracy_svm, "\n")

X_train_final = pd.concat([X_train_full, X_val])
y_train_final = pd.concat([y_train_full, y_val])
svm_final = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", SVC(kernel="linear", C=1.0, random_state=42))
])
svm_final.fit(X_train_final, y_train_final)
y_test_pred_svm = svm_final.predict(X_test)
test_acc_svm = accuracy_score(y_test, y_test_pred_svm)
cm_svm = confusion_matrix(y_test, y_test_pred_svm)
print("=== SVM Final Model (Test Set) ===")
print(f"Test Accuracy: {test_acc_svm:.4f}")
print("Confusion Matrix (rows = true, cols = predicted):")
print(cm_svm, "\n")


Missing values per column:
PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No-show           0
dtype: int64

Shape before: (110527, 14)
Shape after dropping missing values: (110527, 14)
  Gender  Age  Scholarship  Hipertension  Diabetes  Alcoholism  Handcap  \
0      F   62            0             1         0           0        0   
1      M   56            0             0         0           0        0   
2      F   62            0             0         0           0        0   
3      F    8            0             0         0           0        0   
4      F   56            0             1         1           0        0   

   SMS_received  
0             0  
1             0  
2             0  
3             0  
4             0  
Train shape: (88421, 8)
Validation shape: (